In [ ]:
print('Importing libraries...')
import json
import os
import sys
from pathlib import Path
import warnings 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim

from scripts.env import set_deterministic_behaviour
from scripts.dataset import CachedDatasetEncoderPretraining
from scripts.refactoring_caching_single import cache_validation
from scripts.pretraining_metrics import get_balanced_accuracies, get_map
from scripts.lora import inject_lora_into_dinov3_qkv
warnings.filterwarnings("ignore")

PWD = Path.cwd()
print(f"PWD: {PWD}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory Allocated: {torch.cuda.memory_allocated(i) / 1024**2:.1f} MB")
        print(f"  Memory Cached:    {torch.cuda.memory_reserved(i) / 1024**2:.1f} MB")
else:
    device = torch.device("cpu")

seed = 0
set_deterministic_behaviour(seed)

Importing libraries...


In [ ]:
annotations_path = PWD / 'config/reformatted_annotations_single.json'
dataset_dir = PWD.parent / 'dataset/endoscapes/'
cached_images_path = PWD / 'cached_images_multilabel'

# Parameters to create the dataset
combine_train_val = False   # combines train and val subsets for training
remove_black = False        # removes black images from the dataset
roi = False                 # crops to detected ROI
key_frames_only = True      # uses only manually annotated frames
force_recache = False       # recaches the dataset

# These are declared in scripts/refactoring_caching_single.py
# IMAGE_SIZE = (384, 384)
# DATASET_MEAN = (0.454315, 0.290313, 0.299898)
# DATASET_STD = (0.167318, 0.156652, 0.150197)

cache_validation(   dataset_dir,
                    cached_images_path,
                    annotations_path,
                    combine_train_val = combine_train_val,
                    remove_black = remove_black,
                    roi=roi,
                    key_frames_only = key_frames_only,
                    force_recache = force_recache)

Rebuilding cached images...
Dataset summary - total length: 11090. Removed black images? False. Kept only key frames? True. RoI? False.
Dataset split: 6960/2331/1799


Caching centre-crop images: 100%|██████████| 6960/6960 [00:52<00:00, 132.56it/s]


✓ Cached 6960 images to /Users/franek/Documents/python_code/MICCVS/cached_images_multilabel/train


Caching centre-crop images: 100%|██████████| 2331/2331 [00:19<00:00, 118.50it/s]


✓ Cached 2331 images to /Users/franek/Documents/python_code/MICCVS/cached_images_multilabel/val


Caching centre-crop images: 100%|██████████| 1799/1799 [00:15<00:00, 113.64it/s]

✓ Cached 1799 images to /Users/franek/Documents/python_code/MICCVS/cached_images_multilabel/test


In [ ]:
# DATASET / DATALOADER CREATION
# Paths
train_set_path = cached_images_path / 'train'
val_set_path = cached_images_path / 'val'
test_set_path = cached_images_path / 'test'

# Datasets
dataset_train = CachedDatasetEncoderPretraining(train_set_path,
                                label_criterion = (None, 'hard'))
dataset_val = CachedDatasetEncoderPretraining(  val_set_path,
                                label_criterion = (None, 'hard'))
dataset_test = CachedDatasetEncoderPretraining( test_set_path,
                                label_criterion = (None, 'hard'))

# Dataloaders
train_dataloader = DataLoader(  dataset_train,
                                batch_size = 16,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = True)

val_dataloader = DataLoader(    dataset_val,
                                batch_size = 16,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = False)

test_dataloader = DataLoader(   dataset_test,
                                batch_size = 16,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = False)

In [ ]:
# INITIALISATION OF THE MODEL
# Init DinoV3 Backbone
REPO_DIR = PWD.parent / 'dinov3'
sys.path.insert(0, str(REPO_DIR))
dinov3_pretarined_weights = './weights/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'
model = torch.hub.load( REPO_DIR,
                        'dinov3_vitb16',
                        source='local',
                        weights=dinov3_pretarined_weights)

# Classification head
in_feats = 768
model.head = nn.Sequential(
    nn.Linear(in_feats, in_feats),
    nn.GELU(),
    nn.Linear(in_feats, 3)
)

# Freeze whole model
for p in model.parameters():
    p.requires_grad = False

# Inject LoRA
num_blocks = len(model.blocks)
target_layers = list(range(num_blocks - 6, num_blocks))
lora_modules = inject_lora_into_dinov3_qkv(model, r=6, layers=target_layers, verbose=True, alpha = 12)

# Enable gradients only for LoRA A/B and head
for p in model.head.parameters():
    p.requires_grad = True

for module in lora_modules:
    for name, p in module.named_parameters():
        if "linear_a_" in name or "linear_b_" in name:
            p.requires_grad = True

# Separate parameter groups
lora_params = []
head_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if "head" in name:
        head_params.append(param)
    else:
        lora_params.append(param)

model.to(device)

optimizer = optim.AdamW(
    [
        {"params": lora_params, "lr": 1e-4},   # LoRA adapters
        {"params": head_params, "lr": 1e-4},   # Classifier head
    ],
    weight_decay=1e-2)

class_weights = torch.tensor([3.19852941, 4.46153846, 2.79518072]).to(device)
bce_loss = nn.BCEWithLogitsLoss(weight=class_weights).to(device)

[LoRA] Injected into qkv of 6 blocks (rank=6).


In [ ]:
# MODEL TRAINING AND EVALUATION
EPOCHS = 10
exp_name = "DinoV3_encoder_pretraining"
results_dict = {}

best_bacc_across_epochs = -1.0
best_epoch = 0

for epoch in range(EPOCHS):
    print(f"Epoch: {epoch+1:02}/{EPOCHS:02}")

    print("Training")
    train_loss_sum = 0.0
    train_probs = []
    train_preds = []
    train_labels = []
    len_train_loader = len(train_dataloader)

    model.train()
    for idx, (images, labels) in enumerate(train_dataloader):
        print(f'\r{idx+1}/{len_train_loader}', end='', flush=True)
        optimizer.zero_grad()
        images, labels = images.to(device), labels.to(device)

        output = model(images)

        train_loss_per_batch = bce_loss(output, labels)

        train_loss_per_batch.backward()
        optimizer.step()

        train_prob = torch.sigmoid(output)
        train_pred = torch.round(train_prob)

        train_probs.append(train_prob.detach().cpu())
        train_preds.append(train_pred.detach().cpu())
        train_labels.append(labels.detach().cpu())
        train_loss_sum += train_loss_per_batch.item()
        #torch.cuda.synchronize()
        if idx==3: break

    C1_balanced_accuracy, C2_balanced_accuracy, C3_balanced_accuracy, total_balanced_accuracy = get_balanced_accuracies(train_labels, train_preds)
    C1_ap, C2_ap, C3_ap, mAP = get_map(train_labels, train_probs)

    avg_train_loss = train_loss_sum / len_train_loader

    print(f"\n--- Training Metrics ---")
    print(f"Train Balanced Accuracy:   {total_balanced_accuracy:.4f}")
    print(f"Train C1 bacc:             {C1_balanced_accuracy:.4f}")
    print(f"Train C2 bacc:             {C2_balanced_accuracy:.4f}")
    print(f"Train C3 bacc:             {C3_balanced_accuracy:.4f}")
    print(f"Train mAP:                 {mAP:.4f}")
    print(f"Train C1 ap:               {C1_ap:.4f}")
    print(f"Train C2 ap:               {C2_ap:.4f}")
    print(f"Train C3 ap:               {C3_ap:.4f}")
    print(f"Train Loss:                {avg_train_loss:.4f}")
    epoch_results = {   'bacc':     round(total_balanced_accuracy, 4),
                        'c1_bacc':  round(C1_balanced_accuracy, 4),
                        'c2_bacc':  round(C2_balanced_accuracy, 4),
                        'c3_bacc':  round(C3_balanced_accuracy, 4),
                        'map':      round(mAP, 4),
                        'c1_map':   round(C1_ap, 4),
                        'c2_map':   round(C2_ap, 4),
                        'c3_map':   round(C3_ap, 4),
                        'loss':     round(avg_train_loss,4)}
    results_dict[f"Epoch {epoch+1} Train"] = epoch_results

    print('Validation')
    val_loss_sum = 0.0
    val_probs = []
    val_preds = []
    val_labels = []
    len_val_loader = len(val_dataloader)
    model.eval()
    with torch.inference_mode():
        for idx, (images, labels) in enumerate(val_dataloader):
            print(f'\r{idx+1}/{len_val_loader}', end='', flush=True)
            images, labels = images.to(device), labels.to(device)
            output = model(images)

            val_loss_per_batch = bce_loss(output, labels)

            val_prob = torch.sigmoid(output)
            val_pred = torch.round(val_prob)

            val_probs.append(val_prob.detach().cpu())
            val_preds.append(val_pred.detach().cpu())
            val_labels.append(labels.detach().cpu())
            val_loss_sum += val_loss_per_batch.item()
            if idx==3: break
        
        C1_balanced_accuracy, C2_balanced_accuracy, C3_balanced_accuracy, total_balanced_accuracy = get_balanced_accuracies(val_labels, val_preds)
        C1_ap, C2_ap, C3_ap, mAP = get_map(val_labels, val_probs)

        avg_val_loss = val_loss_sum / len_val_loader

        print(f"\n--- Validation Metrics ---")
        print(f"Val Balanced Accuracy:     {total_balanced_accuracy:.4f}")
        print(f"Val C1 bacc:               {C1_balanced_accuracy:.4f}")
        print(f"Val C2 bacc:               {C2_balanced_accuracy:.4f}")
        print(f"Val C3 bacc:               {C3_balanced_accuracy:.4f}")
        print(f"Val mAP:                   {mAP:.4f}")
        print(f"Val C1 ap:                 {C1_ap:.4f}")
        print(f"Val C2 ap:                 {C2_ap:.4f}")
        print(f"Val C3 ap:                 {C3_ap:.4f}")
        print(f"Val Loss:                  {avg_val_loss:.4f}")

        val_preds_2save = torch.cat(val_preds, dim=0).tolist()
        val_probs_2save = torch.cat(val_probs, dim=0).tolist()
        val_labels_2save = torch.cat(val_labels, dim=0).tolist()

        epoch_results = {   'bacc':     round(total_balanced_accuracy, 4),
                            'c1_bacc':  round(C1_balanced_accuracy, 4),
                            'c2_bacc':  round(C2_balanced_accuracy, 4),
                            'c3_bacc':  round(C3_balanced_accuracy, 4),
                            'map':      round(mAP, 4),
                            'c1_map':   round(C1_ap, 4),
                            'c2_map':   round(C2_ap, 4),
                            'c3_map':   round(C3_ap, 4),
                            'loss':     round(avg_val_loss,4),
                            'probs':    val_probs_2save,
                            'preds':    val_preds_2save,
                            'labels':   val_labels_2save}


        results_dict[f"Epoch {epoch+1} Val"] = epoch_results

        # Save results
        with open(PWD / 'results' / f'{exp_name}_results.json', 'w') as file:
            json.dump(results_dict, file, indent=4)

        # Save weights of the best epoch
        if total_balanced_accuracy >= best_bacc_across_epochs:
            best_bacc_across_epochs = total_balanced_accuracy
            best_epoch = epoch+1

            print(f"New best result (Epoch {best_epoch}), saving weights...")
            weights_path = Path.cwd() / 'weights'
            checkpoint_dir = os.path.join(weights_path, f'{exp_name}.pt')
            torch.save(model.state_dict(), checkpoint_dir)
        else:
            print('\n')
            
print(f"Testing @ epoch {best_epoch}")
test_loss_sum = 0.0
test_probs = []
test_preds = []
test_labels = []
len_test_loader = len(test_dataloader)
checkpoint = torch.load(checkpoint_dir, map_location=device)
model.load_state_dict(checkpoint)
model.to(device)
model.eval()
with torch.inference_mode():
    for idx, (images, labels) in enumerate(test_dataloader):
        print(f'\r{idx+1}/{len_test_loader}', end='', flush=True)
        images, labels = images.to(device), labels.to(device)
        output = model(images)

        test_loss_per_batch = bce_loss(output, labels)

        test_prob = torch.sigmoid(output)
        test_pred = torch.round(test_prob)

        test_probs.append(test_prob.detach().cpu())
        test_preds.append(test_pred.detach().cpu())
        test_labels.append(labels.detach().cpu())
        test_loss_sum += test_loss_per_batch.item()
        if idx==3: break


C1_balanced_accuracy, C2_balanced_accuracy, C3_balanced_accuracy, total_balanced_accuracy = get_balanced_accuracies(test_labels, test_preds)
C1_ap, C2_ap, C3_ap, mAP = get_map(test_labels, test_probs)
avg_test_loss = test_loss_sum / len_test_loader

print(f"\n--- Testing Metrics ---")
print(f"Test Balanced Accuracy:     {total_balanced_accuracy:.4f}")
print(f"Test C1 bacc:               {C1_balanced_accuracy:.4f}")
print(f"Test C2 bacc:               {C2_balanced_accuracy:.4f}")
print(f"Test C3 bacc:               {C3_balanced_accuracy:.4f}")
print(f"Test mAP:                   {mAP:.4f}")
print(f"Test C1 ap:                 {C1_ap:.4f}")
print(f"Test C2 ap:                 {C2_ap:.4f}")
print(f"Test C3 ap:                 {C3_ap:.4f}")
print(f"Test Loss:                  {avg_test_loss:.4f}")

test_preds_2save = torch.cat(test_preds, dim=0).tolist()
test_probs_2save = torch.cat(test_probs, dim=0).tolist()
test_labels_2save = torch.cat(test_labels, dim=0).tolist()

epoch_results = {   'bacc':     round(total_balanced_accuracy, 4),
                    'c1_bacc':  round(C1_balanced_accuracy, 4),
                    'c2_bacc':  round(C2_balanced_accuracy, 4),
                    'c3_bacc':  round(C3_balanced_accuracy, 4),
                    'map':      round(mAP, 4),
                    'c1_map':   round(C1_ap, 4),
                    'c2_map':   round(C2_ap, 4),
                    'c3_map':   round(C3_ap, 4),
                    'loss':     round(avg_test_loss,4),
                    'probs':    test_probs_2save,
                    'preds':    test_preds_2save,
                    'labels':   test_labels_2save}

results_dict[f"Epoch {best_epoch} Test"] = epoch_results

with open(PWD / 'results' / f'{exp_name}_results.json', 'w') as file:
    json.dump(results_dict, file, indent=4)


Epoch: 01/01
Training
4/435
--- Training Metrics ---
Train Balanced Accuracy:   0.0889
Train C1 bacc:             0.4981
Train C2 bacc:             0.5000
Train C3 bacc:             0.4823
Train mAP:                 0.2244
Train C1 ap:               0.2447
Train C2 ap:               0.1594
Train C3 ap:               0.2690
Train Loss:                0.0193
Validation
4/146
--- Validation Metrics ---
Val Balanced Accuracy:     nan
Val C1 bacc:               nan
Val C2 bacc:               nan
Val C3 bacc:               nan
Val mAP:                   0.0000
Val C1 ap:                 0.0000
Val C2 ap:                 0.0000
Val C3 ap:                 0.0000
Val Loss:                  0.0435


Testing @ epoch 0


NameError: name 'checkpoint_dir' is not defined